# First-token gap vs full-answer delta

**The question.** For tulu, mean `delta` (eval.py's length-normalised full-answer
log-prob gap, positive = favours the CONTEXT answer) barely moves across the decline:
peak `+1.177` -> trough `+1.093`, with only 54% of items moving toward the parametric
answer -- essentially chance. Yet R_ctx over those same checkpoints drops 0.955 -> 0.776.
So at the trough the model still *scores* the contextual answer higher, but its greedy
output increasingly *states* the parametric one.

**The hypothesis.** `delta` averages over all answer tokens, but greedy generation
commits at the **first token**. An answer can win on average and lose at token 1. If the
inversion lives in first-token commitment, then the first-token gap should track the
R_ctx trajectory (drop peak->trough, recover) while `delta` does not.

**The measurement.** For every item that passed the Layer-0 filter, at each phase
checkpoint, record both quantities on the *same* prompt:

- `delta`      -- `eval.py`'s own `method_logprob` (unchanged, reused directly)
- `first_gap`  -- `logit(cf_first_token) - logit(par_first_token)` at the last prompt
                  position, same sign convention

No judge, no API, forward passes only. ~400 items x 5 checkpoints.

### Repo + Drive + checkpoints

In [ ]:
import os

GITHUB_REPO_URL = "https://github.com/GIRIAYUSH/context-parametric-inversion-research.git"
REPO_DIR = "/content/context-parametric-inversion-research"
BRANCH = "dev"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {GITHUB_REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}
!git log --oneline -3

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Staging Checkpoints from Drive

In [ ]:
import yaml, shutil

cfg = yaml.safe_load(open("src/mechanistic-analysis/checkpoints.yaml"))
STAGE_DIR = "/content/staged_checkpoints"

for run, run_cfg in cfg["runs"].items():
    for phase, ckpt_name in run_cfg["phases"].items():
        if ckpt_name is None:
            continue
        dst = os.path.join(STAGE_DIR, run, ckpt_name)
        if os.path.isdir(dst):
            print("already staged:", run, ckpt_name)
            continue
        print("staging", run, phase, ckpt_name)
        shutil.copytree(os.path.join(run_cfg["drive_checkpoint_dir"], ckpt_name), dst)

def phase_path(run, phase):
    c = cfg["runs"][run]["phases"][phase]
    return None if c is None else os.path.join(STAGE_DIR, run, c)

### Model + tokenizer

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Read from Colab Secrets (key icon in the left sidebar) -- add a secret
# named HF_TOKEN. Never paste the token literal here: it gets committed,
# GitHub push protection blocks the push, and the token has to be rotated.
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

In [ ]:
!pip install -q peft accelerate

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(cfg["base_model"] , token = HF_TOKEN)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    cfg["base_model"], token = HF_TOKEN,torch_dtype=torch.bfloat16, device_map="cuda").eval()
print("loaded", cfg["base_model"])

def model_for_phase(ckpt_path):
    """Swaps the LoRA adapter in place; base weights load once."""
    global base_model
    if isinstance(base_model, PeftModel):
        base_model = base_model.unload()
    if ckpt_path is not None:
        base_model = PeftModel.from_pretrained(base_model, ckpt_path).eval()
    return base_model

In [ ]:
# Llama-2's tokenizer ships no chat template, so eval.py's use_ct=True would silently
# fall back to a raw string the model was never tuned on. Set the tulu format (run_sft.py's
# _concat_messages) and verify byte-for-byte against a prompt the original eval stored.
TULU_CHAT_TEMPLATE = (
    "{{ bos_token }} "          # trailing space: reproduces the sentencepiece prefix
    "{% for message in messages %}"
    "{% if message['role'] == 'user' %}"
    "{{ '<|user|>\n' + message['content'].strip() + '\n' }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ '<|assistant|>\n' + message['content'].strip() + eos_token + '\n' }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|assistant|>\n' }}{% endif %}"
)
tok.chat_template = TULU_CHAT_TEMPLATE

import sys
sys.path.insert(0, "src/evaluation")
import eval as cpi_eval

REF = "results/cpi-results/tulu-results/checkpoint_step3150_metrics.json"
ref_row = next(r for r in json.load(open(REF, encoding="utf-8"))["per_item"]
               if r["item_id"] == "cap_0013")
ours = tok.decode(cpi_eval.build_score_prompt_ids(
    tok, ref_row["question"], ref_row["context"], True), skip_special_tokens=False)

assert ours == ref_row["score_prompt_text"], f"PROMPT MISMATCH\nours  : {ours!r}\nstored: {ref_row['score_prompt_text']!r}"
print("prompt verified -- byte-identical to the recorded eval")

### Items and the reference results

`passed` (the Layer-0 filter) and the stored `delta` come from the committed per-checkpoint results, so we measure exactly the item set the original trajectory used -- and get a free correctness check by comparing our recomputed `delta` against the stored one.

In [ ]:
items = {it["item_id"]: it for it in cpi_eval.load_items("dataset/conflict_eval_unified.json")}

# (run, phase, step, path to the committed result for that step)
PHASES = [
    ("alpaca", "peak",     100,  "results/cpi-results/alpaca-results/checkpoint_step100_metrics.json"),
    ("alpaca", "trough",   800,  "results/cpi-results/alpaca-results/checkpoint_step800_metrics.json"),
    ("tulu",   "peak",     500,  "results/cpi-results/tulu-results/checkpoint_step500_metrics.json"),
    ("tulu",   "trough",   3150, "results/cpi-results/tulu-results/checkpoint_step3150_metrics.json"),
    ("tulu",   "recovery", 5000, "results/cpi-results/tulu-results/checkpoint_step5000_metrics.json"),
]

def reference(path):
    """{item_id: stored delta} for items that passed the Layer-0 filter."""
    rows = json.load(open(path, encoding="utf-8"))["per_item"]
    return {r["item_id"]: r["logprob"]["delta"] for r in rows
            if r.get("passed") and r.get("logprob") and "delta" in r["logprob"]}

for run, phase, step, path in PHASES:
    print(f"{run:7} {phase:9} step{step:<5} passed={len(reference(path))}")

### Measure

In [ ]:
!pip uninstall -q -y torchao

In [ ]:
@torch.no_grad()
def gap_and_argmax(model, prompt_ids, par_tok, cf_tok):
    """Returns (first-token gap, greedy first token).

    gap = logit(cf first token) - logit(par first token) at the last prompt
    position -- same sign convention as eval.py's delta, so > 0 favours CONTEXT.
    The argmax token tells us what the model would actually emit first, which
    matters on the gen prompt: with no prefill it sometimes emits a preamble
    ("Answer:") rather than the answer, in which case the gap is measuring
    formatting rather than the answer choice."""
    logits = model(prompt_ids.unsqueeze(0).to(model.device)).logits[0, -1, :]
    gap = (logits[cf_tok] - logits[par_tok]).item()
    return gap, tok.decode(logits.argmax().item())


def first_tok(text):
    return tok(" " + text.strip(), add_special_tokens=False).input_ids[0]


# The 2x2. delta/first_gap are measured on the SCORE prompt (prefill
# "The answer is:") but every label comes from free generation on the GEN prompt
# (no prefill). Those differ in two ways at once -- the prompt AND the procedure
# (teacher-forced scoring of fixed candidates vs free decoding). Measuring both
# metrics on BOTH prompts isolates the prompt while holding the procedure fixed.
PROMPTS = {"score": cpi_eval.build_score_prompt_ids,
           "gen":   cpi_eval.build_gen_prompt_ids}

rows = []
for run, phase, step, path in PHASES:
    stored = reference(path)
    model = model_for_phase(phase_path(run, phase))
    print(f"measuring {run}/{phase} ({len(stored)} items) ...", flush=True)

    for iid, stored_delta in stored.items():
        it = items[iid]
        par, cf = it["parametric_answer"], it["counterfactual_answer"]
        ptok, ctok = first_tok(par), first_tok(cf)

        rec = {"run": run, "phase": phase, "step": step, "item_id": iid,
               "stored_delta": stored_delta}
        for tag, build in PROMPTS.items():
            pid = build(tok, it["question"], it["context"], True)
            rec[f"delta_{tag}"] = cpi_eval.method_logprob(model, tok, pid, par, cf)["delta"]
            g, first = gap_and_argmax(model, pid, ptok, ctok)
            rec[f"first_gap_{tag}"] = g
            rec[f"first_emitted_{tag}"] = first
        rows.append(rec)

json.dump(rows, open("src/mechanistic-analysis/first_token_gap.json", "w"), indent=2)
print(f"\nwrote src/mechanistic-analysis/first_token_gap.json ({len(rows)} rows)")

### Check we reproduce the original measurement

In [ ]:
import numpy as np

# The stored deltas were produced BEFORE the double-space bug was fixed
# (commit df3b7cd "fix double-space bug"); every result file carries
# logprob_singlespace_correction_applied=False, and __version__ was never bumped,
# so both say 1.0.0. Our delta_score therefore should NOT equal the stored value.
# What must hold is that they rank items the same way -- a high correlation.
# A low correlation means something is actually wrong.
for run, phase, step, _ in PHASES:
    sub = [r for r in rows if r["run"] == run and r["phase"] == phase]
    a = np.array([r["delta_score"] for r in sub])
    b = np.array([r["stored_delta"] for r in sub])
    r_ = np.corrcoef(a, b)[0, 1]
    print(f"{run:7} {phase:9} n={len(sub):4}  corr(ours, stored)={r_:.3f}  "
          f"mean offset={float((a-b).mean()):+.3f}")
print("\ncorr > ~0.95 = we reproduce the recorded measurement up to the known "
      "double-space fix.\ncorr < ~0.9 = investigate before reading anything below.")

### Result: does first_gap track R_ctx where delta doesn't?

In [ ]:
R_CTX = {("alpaca", "peak"): 0.890, ("alpaca", "trough"): 0.789,
         ("tulu", "peak"): 0.955, ("tulu", "trough"): 0.776, ("tulu", "recovery"): 0.902}

M = ["delta_score", "delta_gen", "first_gap_score", "first_gap_gen"]

print("MEANS per checkpoint (> 0 favours CONTEXT)")
print(f"{'run':8}{'phase':10}{'n':>5}" + "".join(f"{m:>17}" for m in M) + f"{'R_ctx':>8}")
summary = {}
for run, phase, step, _ in PHASES:
    sub = [r for r in rows if r["run"] == run and r["phase"] == phase]
    vals = {m: float(np.mean([r[m] for r in sub])) for m in M}
    summary[(run, phase)] = vals
    print(f"{run:8}{phase:10}{len(sub):5}" + "".join(f"{vals[m]:17.3f}" for m in M)
          + f"{R_CTX[(run, phase)]:8.3f}")

print("\nPAIRED per-item shifts across each leg (toward PAR = the CPI direction)")
for run, ep, lp in [("alpaca", "peak", "trough"),
                    ("tulu", "peak", "trough"), ("tulu", "trough", "recovery")]:
    e = {r["item_id"]: r for r in rows if r["run"] == run and r["phase"] == ep}
    l = {r["item_id"]: r for r in rows if r["run"] == run and r["phase"] == lp}
    common = sorted(set(e) & set(l))
    dR = R_CTX[(run, lp)] - R_CTX[(run, ep)]
    print(f"\n  {run} {ep}->{lp}  n={len(common)}   R_ctx change = {dR:+.3f}")
    for m in M:
        d = np.array([l[i][m] - e[i][m] for i in common])
        print(f"     {m:16} mean={d.mean():+7.3f}   toward PAR: {(d<0).sum():4}/{len(d)} ({(d<0).mean():.0%})")

# is first_gap measuring the answer, or formatting?
print("\nHow often is the greedy first token NOT a candidate answer's first token?")
for run, phase, step, _ in PHASES:
    sub = [r for r in rows if r["run"] == run and r["phase"] == phase]
    for tag in ("score", "gen"):
        it0 = [items[r["item_id"]] for r in sub]
        off = sum(1 for r, it in zip(sub, it0)
                  if r[f"first_emitted_{tag}"].strip() not in
                     (it["parametric_answer"].split()[0], it["counterfactual_answer"].split()[0]))
        print(f"  {run:7} {phase:9} {tag:6} {off:4}/{len(sub)} ({off/len(sub):.0%}) emit something else first")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, run in zip(axes, ["alpaca", "tulu"]):
    ph = [p for r, p, _, _ in PHASES if r == run]
    x = range(len(ph))
    for m, style in zip(M, ["o-", "o--", "s-", "s--"]):
        ax.plot(x, [summary[(run, p)][m] for p in ph], style, label=m)
    ax.set_xticks(list(x)); ax.set_xticklabels(ph)
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_ylabel("favours CONTEXT  -->")
    ax2 = ax.twinx()
    ax2.plot(x, [R_CTX[(run, p)] for p in ph], "^:", color="crimson", lw=2, label="R_ctx")
    ax2.set_ylabel("R_ctx", color="crimson")
    ax.set_title(run); ax.legend(loc="upper left", fontsize=8); ax2.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig("src/mechanistic-analysis/first_token_gap.png", dpi=150)
plt.show()

### Reading it

The label (and so R_ctx) comes from **free generation on the gen prompt**. The metrics
that did NOT move were measured on the **score prompt**. Those differ in two ways at
once -- prompt and procedure -- so the 2x2 above separates them:

- **`delta_gen` / `first_gap_gen` shift toward PAR across the tulu decline while the
  `_score` versions don't** -> the prefill `"The answer is:"` was masking the inversion.
  It is genuinely prompt-dependent, and that is a sharp, publishable claim.
- **Nothing shifts on either prompt** -> the inversion is not in the candidate-answer
  logits at all. It lives in free decoding itself, and the next place to look is the
  decoding trajectory (what the model commits to, and when), not candidate scores.
- **Watch the last block**: if the greedy first token is often *neither* candidate's
  first token (a preamble like "Answer:"), then `first_gap` on that prompt is partly
  measuring formatting, and a low shift there is weak evidence either way.

Recall what has to be explained: over the tulu decline the *label* moves hard
(CTX/PAR 381/17 -> 308/89, and the judge-free `ordered` classifier agrees, 0.950 ->
0.771) while `delta_score` is flat. Something has to account for that gap.